In [7]:
import SimpleITK as sitk
import numpy as np
import pyvista as pv
from morphometry.cartilage import knee
from morphometry.image_io import Image

In [8]:
image = Image('nibabel')
image.read_image('/home/simon/Data/Duesseldorf/T1rho/P04/1Relaxed/1Relaxed.nii.gz')

In [9]:
image.transform_coordinate_system()

In [10]:
tibia = knee.Tibia(image, 4)

In [11]:
tibia.get_surface_points()

In [ ]:
p = pv.Plotter()
p.add_mesh(pv.PolyData(tibia.inferior_surface))
p.add_mesh(pv.PolyData(tibia.superior_surface), color='red')
p.show()

In [12]:
tibia.calculate_landmarks()

In [13]:
tibia.extract_subregions()

In [15]:
tibia.left_landmarks

{'center': array([127.0546671 , 142.17052872, 164.58877285]),
 'ellipse': 10.0,
 'corners': {'upper_right': array([102., 156.]),
  'lower_right': array([175., 156.]),
  'upper_left': array([102., 103.]),
  'lower_left': array([175., 103.])}}

In [16]:
tibia.right_landmarks

{'center': array([ 59.72119739, 134.88158621, 171.18038219]),
 'ellipse': 10.0,
 'corners': {'upper_right': array([95., 82.]),
  'lower_right': array([175.,  82.]),
  'upper_left': array([95., 32.]),
  'lower_left': array([175.,  32.])}}

In [14]:
p = pv.Plotter()
colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'cyan', 'magenta', 'grey', 'black']
for i, subregion in enumerate(['clt', 'ilt', 'elt', 'alt', 'plt', 'crt', 'irt', 'ert', 'art', 'prt']):
    if len(getattr(tibia, subregion)) == 0:
        continue
    superior, inferior = knee.get_superior_and_inferior_surface_points(getattr(tibia, subregion))
    p.add_mesh(pv.PolyData(superior * np.array([1, 1, 1.2])), color=colors[i])
    p.add_mesh(pv.PolyData(inferior), color=colors[i])

p.enable_eye_dome_lighting()
p.show()

Widget(value='<iframe src="http://localhost:32921/index.html?ui=P_0x71a97ada38b0_0&reconnect=auto" class="pyvi…

In [ ]:
tibia.calculate_thickness(method='knn')

In [ ]:
femur = knee.Femur(image, 3)
femur.calculate_thickness(tibia)

In [ ]:
p = pv.Plotter()
colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'cyan', 'magenta', 'grey', 'black']
for i, subregion in enumerate(['left_cwbz', 'right_cwbz', 'left_posterior_zone', 'right_posterior_zone', 'left_anterior_zone', 'right_anterior_zone']):
    if len(getattr(femur, subregion)) == 0:
        continue
    if subregion in ['left_posterior_zone', 'right_posterior_zone']:
        tmp = getattr(femur, subregion).copy()
        tmp[:, 1], tmp[:, 2] = tmp[:, 2], tmp[:, 1].copy()  # rotate to allow extraction of anterior and posterior surface
        superior_surface, inferior_surface = knee.get_superior_and_inferior_surface_points(tmp)
        superior_surface[:, 1], superior_surface[:, 2] = superior_surface[:, 2], superior_surface[:, 1].copy()  # rotate back
        inferior_surface[:, 1], inferior_surface[:, 2] = inferior_surface[:, 2], inferior_surface[:, 1].copy()
    else:
        superior_surface, inferior_surface = knee.get_superior_and_inferior_surface_points(getattr(femur, subregion))

    p.add_mesh(pv.PolyData(superior_surface * np.array([1, 1, 1.2])), color=colors[i])
    p.add_mesh(pv.PolyData(inferior_surface), color=colors[i])

p.show()

In [ ]:
p = pv.Plotter()
tmp = getattr(femur, 'left_posterior_zone').copy()
tmp[:, 2], tmp[:, 1] = tmp[:, 1], tmp[:, 2].copy()
p.add_mesh(pv.PolyData(tmp))
p.enable_eye_dome_lighting()
p.show()

In [ ]:
superior_surface, inferior_surface = knee.get_superior_and_inferior_surface_points(tmp)
superior_surface[:, 1], superior_surface[:, 2] = superior_surface[:, 2], superior_surface[:, 1].copy()  # rotate back
inferior_surface[:, 1], inferior_surface[:, 2] = inferior_surface[:, 2], inferior_surface[:, 1].copy()

In [ ]:
p = pv.Plotter()
p.add_mesh(pv.PolyData(superior_surface))
p.add_mesh(pv.PolyData(inferior_surface), color='red')
p.enable_eye_dome_lighting()
p.show()

In [ ]:
import SimpleITK as sitk

def register(fixed_image, moving_image):
    translation_transform = sitk.TranslationTransform(fixed_image.GetDimension())
    versor_transform = sitk.VersorRigid3DTransform()
    scale_transform = sitk.ScaleTransform(fixed_image.GetDimension())
    composite_transform = sitk.CompositeTransform(fixed_image.GetDimension())
    # composite_transform.AddTransform(scale_transform)
    # composite_transform.AddTransform(versor_transform)
    composite_transform.AddTransform(translation_transform)

    initial_transform = sitk.CenteredTransformInitializer(
        fixed_image,
        moving_image,
        sitk.Similarity3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )

    # Set up the registration method.
    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMeanSquares()

    registration_method.SetOptimizerAsRegularStepGradientDescent(
        learningRate=0.05,
        minStep=1e-4,
        numberOfIterations=2000,
        gradientMagnitudeTolerance=1e-8
    )
    registration_method.SetInitialTransform(initial_transform)
    registration_method.SetInterpolator(sitk.sitkNearestNeighbor)  # Use nearest neighbor to preserve label values.

    # Execute the registration.
    final_transform = registration_method.Execute(fixed_image, moving_image)
    print(registration_method.GetOptimizerStopConditionDescription())
    print(registration_method.GetOptimizerIteration())
    print(registration_method.GetMetricValue())
    print(registration_method.GetMetricNumberOfValidPoints())

    # print("Optimizer's stopping condition:", registration_method.GetStopConditionDescription())

    # Resample the moving image using the final transform.
    registered_moving = sitk.Resample(
        moving_image,
        fixed_image,
        final_transform,
        sitk.sitkNearestNeighbor,  # ensures labels aren’t smoothed by interpolation
        0.0,
        moving_image.GetPixelIDValue()
    )

    return registered_moving

In [ ]:

import numpy as np
from pathlib import Path
from tqdm import tqdm

for patient in Path('/home/simon/Data/Duesseldorf/T1rho/').iterdir():
    if not patient.is_dir():
        continue

    relaxed_image = sitk.ReadImage(f'/home/simon/Data/Duesseldorf/T1rho/{patient.name}/1Relaxed/1Relaxed.nii.gz')
    med_to_lat_image = sitk.ReadImage(f'/home/simon/Data/Duesseldorf/T1rho/{patient.name}/2MedToLat/2MedToLat.nii.gz')
    lat_to_med_image = sitk.ReadImage(f'/home/simon/Data/Duesseldorf/T1rho/{patient.name}/3LatToMed/3LatToMed.nii.gz')

    relaxed_image = sitk.Cast(relaxed_image, sitk.sitkFloat32)
    lat_to_med_image = sitk.Cast(lat_to_med_image, sitk.sitkFloat32)
    med_to_lat_image = sitk.Cast(med_to_lat_image, sitk.sitkFloat32)

    print(patient.name)
    registered_lat_to_med = register(relaxed_image, lat_to_med_image)
    print('lat_to_med')
    registered_med_to_lat = register(relaxed_image, med_to_lat_image)
    print('med_to_lat')

    sitk.WriteImage(registered_med_to_lat, f'/home/simon/Data/Duesseldorf/T1rho/{patient.name}/2MedToLat/2MedToLat_registered.nii.gz')
    sitk.WriteImage(registered_lat_to_med, f'/home/simon/Data/Duesseldorf/T1rho/{patient.name}/3LatToMed/3LatToMed_registered.nii.gz')

In [ ]:
relaxed_image = sitk.ReadImage(f'/home/simon/Data/Duesseldorf/T1rho/P04/1Relaxed/1Relaxed.nii.gz')
moi = sitk.ReadImage('/home/simon/Data/Duesseldorf/T1rho/P04/3LatToMed/3LatToMed.nii.gz')
moi = sitk.ReadImage('/home/simon/Data/Duesseldorf/T1rho/P04/2MedToLat/2MedToLat.nii.gz')
relaxed_image = sitk.Cast(relaxed_image, sitk.sitkFloat32)
moi = sitk.Cast(moi, sitk.sitkFloat32)
register(relaxed_image, moi)
